# Phase 1: Basic Cleaning


In [1]:
# 1.1 import raw data
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\chennai1-pg-price-predictor\Data\raw\chennai_pg_dataset.csv')

In [2]:
df.shape

(1661, 39)

In [3]:
# 1.2 Deduplication
before = df.shape[0]

# id + occupancy combination vachu dedup pannuthu, full row vachu illa
df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')

after = df.shape[0]
print(f"{before - after} duplicate rows removed")
print("Shape after dedup:", df.shape)


130 duplicate rows removed
Shape after dedup: (1531, 39)


In [4]:
# Drop Unnecessary Columns
drop_cols = [
    'id', 'title', 'address', 'total_bathrooms',
    'gate_closing_time', 'warden', 'cooking_allowed',
    'guardian_required', 'nonveg_allowed', 'smoking_allowed'
]

existing = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping columns:", df.shape)

Dropped columns: ['id', 'title', 'address', 'total_bathrooms', 'gate_closing_time', 'warden', 'cooking_allowed', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
Shape after dropping columns: (1531, 29)


In [5]:
# 1.4 Drop Redundant Food Columns
food_cols = ['breakfast', 'lunch', 'dinner']

existing = [c for c in food_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping food columns:", df.shape)

Dropped columns: ['breakfast', 'lunch', 'dinner']
Shape after dropping food columns: (1531, 26)


In [6]:
# remove rpws
df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
df = df[df['rent'] >= 1000]

df.shape


(1436, 26)

In [7]:
# Fix data types
amenity_cols = [
    'attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
    'refrigerator', 'common_tv', 'room_cleaning', 'room_ac',
    'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
    'room_attached_bath'
]

# Check unique values before converting (safety check)
for col in amenity_cols:
    print(col, df[col].unique())


attached_bathroom [False True]
mess [nan False True]
wifi [nan True False]
laundry [nan True False]
power_backup [nan False True]
refrigerator [nan True False]
common_tv [nan True False]
room_cleaning [nan True False]
room_ac [False True nan]
room_cupboard [False True nan]
room_tv [False True nan]
room_geyser [False True nan]
room_bedding [False True nan]
room_attached_bath [False True nan]


In [8]:
# Missing values-a False nu fill pannunga (amenity available illa nu treat pannuthu)
df[amenity_cols] = df[amenity_cols].fillna(False)

# dtype-a bool aa convert pannunga
df[amenity_cols] = df[amenity_cols].astype(bool)

print("Shape after fixing amenity columns:", df.shape)
df[amenity_cols].dtypes

Shape after fixing amenity columns: (1436, 26)


C:\Users\phari\AppData\Local\Temp\ipykernel_13104\549647577.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[amenity_cols] = df[amenity_cols].fillna(False)


attached_bathroom     bool
mess                  bool
wifi                  bool
laundry               bool
power_backup          bool
refrigerator          bool
common_tv             bool
room_cleaning         bool
room_ac               bool
room_cupboard         bool
room_tv               bool
room_geyser           bool
room_bedding          bool
room_attached_bath    bool
dtype: object

In [9]:
# 1.7 — Replace -10 sentinel with NaN
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# Step Create a missingness indicator (before filling)
df['transit_score_missing'] = df['transit_score'].isna()

#  Fill missing values using locality-level median
df['transit_score'] = df.groupby('locality')['transit_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median = df['transit_score'].median()
df['transit_score'] = df['transit_score'].fillna(overall_median)

print("Remaining missing transit_score:", df['transit_score'].isna().sum())
print("transit_score_missing counts:\n", df['transit_score_missing'].value_counts())

Remaining missing transit_score: 0
transit_score_missing counts:
 transit_score_missing
False    759
True     677
Name: count, dtype: int64


d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [10]:
# 1.8 Create a missingness indicator (before filling)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna()


df['lifestyle_score'] = df.groupby('locality')['lifestyle_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median_lifestyle = df['lifestyle_score'].median()
df['lifestyle_score'] = df['lifestyle_score'].fillna(overall_median_lifestyle)

print("Remaining missing lifestyle_score:", df['lifestyle_score'].isna().sum())
print("lifestyle_score_missing counts:\n", df['lifestyle_score_missing'].value_counts())

d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Remaining missing lifestyle_score: 0
lifestyle_score_missing counts:
 lifestyle_score_missing
False    762
True     674
Name: count, dtype: int64


In [11]:
# Missing parking values-a 'none' category aa fill pannunga
df['parking'] = df['parking'].fillna('none')

print("parking value counts:\n", df['parking'].value_counts())

parking value counts:
 parking
Bike            1156
Bike and Car     137
none             112
Car               31
Name: count, dtype: int64


In [12]:
df['available_for'] = df['available_for'].replace('Both', 'Anyone')
print(df['available_for'].value_counts())

available_for
Anyone                  1307
Working Professional     121
Student                    8
Name: count, dtype: int64


In [13]:
# Final check
print("Final shape:", df.shape)
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicates:", df.duplicated().sum())

Final shape: (1436, 28)

Missing values:
 Series([], dtype: int64)

Duplicates: 0


# Phase 2: Preprocessing

## Split the data

In [14]:
from sklearn.model_selection import train_test_split

X = df.drop('rent', axis=1)
y = df['rent']

## Train Data

In [15]:
#First 80% Train + 20% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [16]:
# Then temporary 20%-a 50/50 split 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


# Identify Numerical & ategorical Features

###  why?
#####    -we  use different preprocessimg methods for categorical and numerical features.
   ##### -Numerical → Scaling / Transformation
 #####   -Categorical → Encoding

In [17]:
# cell 1 use X_train
numeric_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = X_train.select_dtypes(
    include=['object']
).columns

# Ordinal encoding

In [30]:
from sklearn.preprocessing import OrdinalEncoder

occupancy_encoder = OrdinalEncoder(
    categories=[['SINGLE', 'DOUBLE', 'THREE', 'FOUR']]
)

X_train['occupancy_encoded'] = occupancy_encoder.fit_transform(
    X_train[['occupancy']]
)

X_val['occupancy_encoded'] = occupancy_encoder.transform(
    X_val[['occupancy']]
)

X_test['occupancy_encoded'] = occupancy_encoder.transform(
    X_test[['occupancy']]
)

In [31]:
print(X_train[['occupancy', 'occupancy_encoded']].head())
print(X_val[['occupancy', 'occupancy_encoded']].head())
print(X_test[['occupancy', 'occupancy_encoded']].head())

     occupancy  occupancy_encoded
982      THREE                2.0
962     SINGLE                0.0
340      THREE                2.0
1295     THREE                2.0
36      SINGLE                0.0
     occupancy  occupancy_encoded
608     SINGLE                0.0
301     DOUBLE                1.0
1438      FOUR                3.0
83      DOUBLE                1.0
242      THREE                2.0
     occupancy  occupancy_encoded
139       FOUR                3.0
265       FOUR                3.0
1650    DOUBLE                1.0
1451     THREE                2.0
1071    DOUBLE                1.0


In [72]:
X_train = X_train.drop(columns=['occupancy'])
X_val = X_val.drop(columns=['occupancy'])
X_test = X_test.drop(columns=['occupancy'])

In [32]:
# Identify categorical columns
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print(categorical_cols)

['locality', 'gender', 'available_for', 'occupancy', 'parking']


In [34]:
for col in ['gender', 'available_for', 'parking']:
    print(f"\n===== {col} =====")
    print(X_train[col].value_counts())
    print("Categories:", X_train[col].nunique())


===== gender =====
gender
MALE      576
FEMALE    540
BOTH       32
Name: count, dtype: int64
Categories: 3

===== available_for =====
available_for
Anyone                  1045
Working Professional      95
Student                    8
Name: count, dtype: int64
Categories: 3

===== parking =====
parking
Bike            933
Bike and Car    102
none             89
Car              24
Name: count, dtype: int64
Categories: 4


# One-Hot Encoding

In [35]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

In [36]:
cat_cols = ['gender','available_for','parking']

train_encoded = ohe.fit_transform(X_train[cat_cols])

In [37]:
val_encoded = ohe.transform(X_val[cat_cols])

test_encoded = ohe.transform(X_test[cat_cols])

In [39]:
print(train_encoded.shape)
print(val_encoded.shape)
print(test_encoded.shape)

print(encoded_cols)

(1148, 10)
(144, 10)
(144, 10)
['gender_BOTH' 'gender_FEMALE' 'gender_MALE' 'available_for_Anyone'
 'available_for_Student' 'available_for_Working Professional'
 'parking_Bike' 'parking_Bike and Car' 'parking_Car' 'parking_none']


In [40]:
train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_cols,
    index=X_train.index
)

val_encoded_df = pd.DataFrame(
    val_encoded,
    columns=encoded_cols,
    index=X_val.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_cols,
    index=X_test.index
)

In [41]:
# remove the original columns
X_train = X_train.drop(columns=cat_cols)
X_val = X_val.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)

In [42]:
# add the encoded columns
X_train = pd.concat([X_train, train_encoded_df], axis=1)
X_val = pd.concat([X_val, val_encoded_df], axis=1)
X_test = pd.concat([X_test, test_encoded_df], axis=1)

In [43]:
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (1148, 35)
Validation shape: (144, 35)
Test shape: (144, 35)


# Smoothed Target Encoding

In [44]:
locality_stats = X_train.copy()
locality_stats['rent'] = y_train

locality_stats = locality_stats.groupby('locality')['rent'].agg(
    ['mean', 'count']
)

print(locality_stats.head(10))

                                       mean  count
locality                                          
Adambakkam                      6480.769231     13
Alandur                         7557.142857     14
Arcot Road-Kodambakkam          5450.000000      7
Ashok Nagar                     6550.000000      4
Balaji Nagar                    7722.222222      9
East Coast Road-Thiruvanmiyur   8694.444444     36
East Tambaram                   7460.000000     15
GST Road-Tambaram               8318.181818     33
Guindy                          7974.358974     39
Indian Institute Of Technology  6666.666667      3


In [45]:
global_mean = y_train.mean()

print("Global mean rent:", global_mean)

Global mean rent: 7638.965156794425


In [47]:
# smooth encoding
alpha = 10

locality_stats['smoothed_mean'] = (
    (locality_stats['count'] * locality_stats['mean']
     + alpha * global_mean)
    / (locality_stats['count'] + alpha)
)

print(locality_stats.head(10))

                                       mean  count  smoothed_mean
locality                                                         
Adambakkam                      6480.769231     13    6984.332677
Alandur                         7557.142857     14    7591.235482
Arcot Road-Kodambakkam          5450.000000      7    6737.626563
Ashok Nagar                     6550.000000      4    7327.832255
Balaji Nagar                    7722.222222      9    7678.402714
East Coast Road-Thiruvanmiyur   8694.444444     36    8464.992425
East Tambaram                   7460.000000     15    7531.586063
GST Road-Tambaram               8318.181818     33    8160.224455
Guindy                          7974.358974     39    7905.911256
Indian Institute Of Technology  6666.666667      3    7414.588582


In [48]:
locality_mapping = locality_stats['smoothed_mean'].to_dict()

X_train['locality_encoded'] = X_train['locality'].map(
    locality_mapping
)

In [49]:
X_val['locality_encoded'] = X_val['locality'].map(
    locality_mapping
)

X_val['locality_encoded'] = X_val['locality_encoded'].fillna(
    global_mean
)


In [50]:
X_test['locality_encoded'] = X_test['locality'].map(
    locality_mapping
)

X_test['locality_encoded'] = X_test['locality_encoded'].fillna(
    global_mean
)

In [52]:
# remove original locality
X_train = X_train.drop(columns=['locality'])
X_val = X_val.drop(columns=['locality'])
X_test = X_test.drop(columns=['locality'])

# Numerical Preprocessing

In [61]:
numerical_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print(numerical_cols)

['latitude', 'longitude', 'transit_score', 'lifestyle_score', 'deposit', 'occupancy_encoded', 'gender_BOTH', 'gender_FEMALE', 'gender_MALE', 'available_for_Anyone', 'available_for_Student', 'available_for_Working Professional', 'parking_Bike', 'parking_Bike and Car', 'parking_Car', 'parking_none', 'locality_encoded']


## Scaling

In [63]:
numeric_cols = [
    'latitude',
    'longitude',
    'transit_score',
    'lifestyle_score',
    'deposit',
    'locality_encoded'
]

In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [65]:
X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

In [66]:
# validation
X_val[numeric_cols] = scaler.transform(
    X_val[numeric_cols]
)

# test
X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

In [67]:
print(X_train[numeric_cols].describe().T)

                   count          mean       std       min       25%  \
latitude          1148.0 -1.128327e-14  1.000436 -1.643354 -0.964540   
longitude         1148.0 -2.679080e-13  1.000436 -2.905781  0.060446   
transit_score     1148.0  4.951517e-17  1.000436 -4.404772 -0.700001   
lifestyle_score   1148.0 -1.237879e-16  1.000436 -5.122056 -0.504688   
deposit           1148.0  1.485455e-16  1.000436 -0.885619 -0.398702   
locality_encoded  1148.0 -2.258356e-15  1.000436 -3.284022 -0.398279   

                       50%       75%       max  
latitude          0.348304  0.512882  3.254983  
longitude         0.419055  0.536155  1.117908  
transit_score     0.411430  0.707811  1.448765  
lifestyle_score  -0.406446  0.674215  2.835536  
deposit          -0.398702 -0.073982  8.855804  
locality_encoded -0.147031  0.568176  2.176917  


In [73]:
print("Object columns:",
      X_train.select_dtypes(include=['object']).columns.tolist())

Object columns: []
